[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/04_rag_regulation/04_rag_regulation_solutions.ipynb)

# 04. `rag-regulation-example` 동행 — 연습 문제 해설

> 본문: [04_rag_regulation.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/04_rag_regulation/04_rag_regulation.ipynb)

먼저 직접 풀어본 뒤에 보세요. 특히 2번(xref 확장)과 3번(파서 깨뜨리기)은
직접 시도해봐야 얻는 게 있습니다.

## 0. 환경 준비 — 프로젝트를 옆에 펼쳐두기

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

if IN_COLAB:
    # 이 노트북은 "예제 프로젝트를 옆에 두고 같이 읽는" 노트북입니다.
    # 그래서 설명만 하지 않고, 저장소를 통째로 내려받아 **실제 프로젝트 파일**을 열어봅니다.
    subprocess.run(["git", "clone", "-q", "https://github.com/karzit/temp.git", "/content/temp"], check=False)
    REPO_ROOT = "/content/temp"
    !pip install -q langchain-core scikit-learn python-dotenv
else:
    # 로컬에서 열었다면 이 노트북 위치(notebooks/project-walkthrough/NN_xxx/)에서 3단계 위가 저장소 루트입니다.
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

PROJECT = os.path.join(REPO_ROOT, "example-projects", "rag-regulation-example")
SRC = os.path.join(PROJECT, "src")
print("프로젝트 경로:", PROJECT)
assert os.path.isdir(SRC), "프로젝트 경로를 찾지 못했습니다. 저장소 루트에서 노트북을 열었는지 확인하세요."

아래 `show()`는 이 노트북 전체에서 쓰는 도우미입니다. **설명 대신 진짜 프로젝트 파일을 그대로 출력**해서, 노트북과 코드가 어긋나지 않게 합니다.

In [ ]:
import re


def show(filename, start=None, end=None, grep=None):
    """프로젝트 파일의 실제 소스를 줄 번호와 함께 출력한다.

    설명을 읽는 것과 실제 코드를 보는 것 사이의 간격을 없애기 위한 도우미입니다.
    이 노트북에서 "코드 읽기"라고 나오는 곳은 전부 진짜 프로젝트 파일을 그대로 보여줍니다.

        show("crawl.py")                  전체
        show("crawl.py", 30, 45)          30~45번째 줄
        show("crawl.py", grep="def ")     'def '가 들어간 줄만
    """
    path = os.path.join(SRC, filename) if not os.path.isabs(filename) else filename
    lines = open(path, encoding="utf-8").read().splitlines()

    if grep:
        picked = [(i, l) for i, l in enumerate(lines, 1) if re.search(grep, l)]
    else:
        s = (start or 1) - 1
        e = end or len(lines)
        picked = [(i, l) for i, l in enumerate(lines[s:e], s + 1)]

    for i, line in picked:
        print(f"{i:>4} | {line}")


def show_file(relpath, **kwargs):
    """프로젝트 루트 기준 경로로 파일을 보여준다 (README, docker-compose 등)."""
    show(os.path.join(PROJECT, relpath), **kwargs)


# 프로젝트 소스를 import할 수 있도록 경로를 등록해둡니다.
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [ ]:
import json
import re

os.environ.setdefault("OPENAI_API_KEY", "sk-dummy-not-used")

from langchain_core.documents import Document
from parse import parse_articles, split_paragraphs
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

regulation_text = open(os.path.join(PROJECT, "data", "sample_regulation.txt"), encoding="utf-8").read()
golden = json.loads(open(os.path.join(PROJECT, "data", "golden_set.json"), encoding="utf-8").read())

articles = parse_articles([regulation_text])
by_number = {a.number: a for a in articles}
print(f"조 {len(articles)}개, 골든셋 {len(golden['questions'])}문항 준비 완료")

In [ ]:
def build_chunks(max_chars=900):
    docs = []
    for a in articles:
        for marker, body in split_paragraphs(a, max_chars=max_chars):
            docs.append(
                Document(
                    page_content=f"{a.full_path}\n{body}",
                    metadata={"source": "s.txt", "page": a.page, "article": a.number, "paragraph": marker},
                )
            )
    return docs


def make_searcher(chunks):
    corpus = [c.page_content for c in chunks]
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    mat = vec.fit_transform(corpus)

    def search(question, k=5):
        scores = cosine_similarity(vec.transform([question]), mat)[0]
        return [(chunks[i], scores[i]) for i in scores.argsort()[::-1][:k]]

    return search


def evaluate(search_fn, questions, k=5):
    hits = reciprocal = recall_sum = 0.0
    misses = []
    for item in questions:
        gold = set(item["articles"])
        rank_found, covered = 0, set()
        for rank, (chunk, _s) in enumerate(search_fn(item["question"], k=k), 1):
            found = {chunk.metadata.get("article")} & gold
            covered |= found
            if found and not rank_found:
                rank_found = rank
        if rank_found:
            hits += 1
            reciprocal += 1 / rank_found
        else:
            misses.append(item["question"])
        recall_sum += len(covered) / len(gold)
    n = len(questions)
    return {"hit": hits / n, "MRR": reciprocal / n, "recall": recall_sum / n, "misses": misses}


chunks = build_chunks()
search = make_searcher(chunks)
questions = golden["questions"]
base = evaluate(search, questions, k=3)
print(f"기준선 (k=3): hit={base['hit']:.3f} MRR={base['MRR']:.3f} recall={base['recall']:.3f}")
print(f"놓친 질문: {base['misses']}")

## 연습 1. 골든셋 늘리기

**문제**: 5문항을 추가하되 **일부러 어렵게** 만드세요.

**어려운 질문이란 무엇인가**부터 정해야 합니다. 그냥 애매한 질문이 아니라,
**실제 사용자가 던지지만 지금 시스템이 놓치기 쉬운 질문**입니다. 네 종류가 있습니다.

In [ ]:
new_questions = [
    {
        # ① 규정 용어와 일상 용어가 다른 경우 — 검색의 가장 흔한 실패
        "question": "몸이 아파서 며칠 못 나오면 월급은 나오나요?",
        "articles": ["제15조"],
        "type": "용어불일치",  # 규정은 "병가", 사용자는 "몸이 아파서"
    },
    {
        # ② 여러 조를 조합해야 답이 되는 경우
        "question": "야근이랑 밤샘을 같이 하면 수당이 어떻게 붙나요?",
        "articles": ["제12조"],
        "type": "다중항",  # 제12조 ②③④를 다 봐야 함
    },
    {
        # ③ 조건부 질문 — 단서 조항까지 봐야 정확
        "question": "재택근무하다가 야근했는데 미리 말 안 했으면 수당 못 받나요?",
        "articles": ["제11조", "제12조"],
        "type": "상호참조",  # 제11조 ③의 "다만" 단서 + 제12조
    },
    {
        # ④ 부정형 질문 — "안 되는 경우"를 묻는 질문은 검색이 특히 약하다
        "question": "이 규칙이 적용 안 되는 사람도 있나요?",
        "articles": ["제2조"],
        "type": "부정형",
    },
    {
        # ⑤ 규정에 아예 없는 내용 — "모른다"고 답해야 정상
        "question": "출장 갈 때 항공료는 얼마까지 지원되나요?",
        "articles": [],
        "type": "정답없음",
    },
]

for q in new_questions:
    print(f"[{q['type']:<8}] {q['question']}")
    print(f"             정답: {q['articles'] or '(없음 — 모른다고 답해야 함)'}")

> 💡 **⑤번이 특히 중요합니다.** 골든셋에 "정답이 없는 질문"을 넣어야
> 시스템이 **아무거나 그럴듯한 걸 가져오는지** 확인할 수 있습니다.
> 검색은 항상 뭔가를 돌려주기 때문에, 이런 질문이 없으면 "억지로 찾아오는 문제"가 안 보입니다.
>
> 다만 `articles`가 빈 목록이면 `recall`의 분모가 0이 됩니다. 채점 코드를 손봐야 합니다.

In [ ]:
def evaluate_v2(search_fn, questions, k=3):
    """정답이 없는 질문(articles=[])을 함께 채점하는 버전.

    정답 없는 질문은 '낮은 점수로 아무것도 확신하지 않았는지'를 봐야 하므로,
    hit/MRR 집계에서는 빼고 따로 센다. 분모가 0이 되는 것도 이렇게 피한다.
    """
    answerable = [q for q in questions if q["articles"]]
    unanswerable = [q for q in questions if not q["articles"]]

    scores = evaluate(search_fn, answerable, k=k) if answerable else {}

    # 정답 없는 질문에서는 최고 유사도가 낮게 나와야 정상이다.
    top_scores = [search_fn(q["question"], k=1)[0][1] for q in unanswerable]
    scores["정답없음_최고유사도"] = sum(top_scores) / len(top_scores) if top_scores else None
    return scores


combined = questions + new_questions
result = evaluate_v2(search, combined, k=3)
print(f"확장 골든셋 {len(combined)}문항")
print(f"  hit={result['hit']:.3f}  MRR={result['MRR']:.3f}  recall={result['recall']:.3f}")
print(f"  정답 없는 질문의 평균 최고 유사도: {result['정답없음_최고유사도']:.3f}")
print(f"\n  놓친 질문:")
for q in result["misses"]:
    print(f"    - {q}")

**결과를 읽는 법**

- 문항을 늘리면 점수는 **대체로 떨어집니다.** 정상입니다. 쉬운 문제만 있던 시험에 어려운 문제를 넣었으니까요.
  중요한 건 절대값이 아니라 **바꾸기 전후의 변화**입니다.
- "정답 없음"의 최고 유사도가 답 있는 질문과 비슷하게 높다면 위험 신호입니다.
  시스템이 **아무거나 가져와서 그럴듯하게 답할** 준비가 되어 있다는 뜻이니까요.
  실제 서비스에서는 이 값에 문턱을 두고 "관련 규정을 찾지 못했습니다"로 빠지게 만듭니다.

## 연습 2. 상호참조(xref) 확장

**문제**: 제11조 ③항의 "제12조에 따른 수당"을 따라가 제12조도 함께 가져오세요.

**왜 필요한가**: 규정 조문은 서로를 참조합니다. 제11조만 읽으면
"제12조에 따른 수당을 지급한다"까지만 알 수 있고, **얼마인지는 모릅니다.**
사용자가 원하는 답은 "50% 가산"인데 그건 제12조에 있습니다.

### 1단계 — 본문에서 참조를 뽑아내기

In [ ]:
# "제12조", "제9조 제2항", "제11조의2" 같은 참조를 모두 잡는다.
XREF_RE = re.compile(r"제\s*(\d+)\s*조(?:\s*의\s*(\d+))?")


def extract_xrefs(article) -> list[str]:
    """조 본문에서 다른 조를 가리키는 참조를 뽑아낸다.

    자기 자신을 가리키는 참조는 제외한다. 조 표지("제11조(재택근무)")가 본문 맨 앞에
    들어 있어서, 안 빼면 모든 조가 자기 자신을 참조하는 것으로 잡힌다.
    """
    refs = []
    for main, branch in XREF_RE.findall(article.body):
        number = f"제{main}조의{branch}" if branch else f"제{main}조"
        if number != article.number and number not in refs:
            refs.append(number)
    return refs


print("조별 상호참조:")
for a in articles:
    refs = extract_xrefs(a)
    if refs:
        print(f"  {a.number:<10} -> {refs}")

제11조가 제9조와 제12조를 참조하고, 제3조가 제9조·제11조를 참조하는 게 보입니다.
**이게 문서의 숨은 연결 구조입니다.**

### 2단계 — 검색 결과를 1홉 확장하기

여기서 설계 판단이 필요합니다. **참조를 무한정 따라가면 안 됩니다.**
제11조 → 제12조 → 또 다른 조... 이렇게 번지면 결국 규정집 전체가 딸려옵니다.
그래서 **1홉만**, 그리고 **토큰 예산 안에서** 우선순위를 정해 넣습니다.

In [ ]:
xref_map = {a.number: extract_xrefs(a) for a in articles}
chunk_by_article = {}
for c in chunks:
    chunk_by_article.setdefault(c.metadata["article"], c)


def search_with_xref(question, k=3, budget=3):
    """직접 검색 결과에 1홉 참조를 더한다.

    우선순위: 직접 검색 결과 > 그것이 명시적으로 참조하는 조.
    budget으로 총 개수를 제한해서, 확장이 원래 결과를 밀어내지 않게 한다.
    """
    direct = [chunk for chunk, _score in search(question, k=k)]
    result = list(direct)
    seen = {c.metadata["article"] for c in direct}

    for chunk in direct:
        if len(result) >= k + budget:
            break
        for ref in xref_map.get(chunk.metadata["article"], []):
            if ref not in seen and ref in chunk_by_article and len(result) < k + budget:
                result.append(chunk_by_article[ref])
                seen.add(ref)

    return [(c, 0.0) for c in result]


q = "재택근무하다가 야근했는데 수당이 얼마나 붙나요?"
print(f"질문: {q}\n")
print("확장 전:")
for c, s in search(q, k=3):
    print(f"  [{s:.3f}] {c.metadata['article']}")
print("\n확장 후:")
for c, _ in search_with_xref(q, k=3):
    print(f"  {c.metadata['article']}")

In [ ]:
# 참조가 실제로 정답률을 올렸는지 채점해봅니다.
xref_questions = questions + [
    {"question": "재택근무 중 야근수당은 얼마나 가산되나요?", "articles": ["제11조", "제12조"], "type": "상호참조"},
    {"question": "재택근무일에 하루 몇 시간 일하나요?", "articles": ["제11조", "제9조"], "type": "상호참조"},
]

before = evaluate(search, xref_questions, k=3)
after = evaluate(search_with_xref, xref_questions, k=3)

print(f"{'':<12}{'hit':>8}{'recall':>10}")
print("-" * 30)
print(f"{'확장 전':<12}{before['hit']:>8.3f}{before['recall']:>10.3f}")
print(f"{'확장 후':<12}{after['hit']:>8.3f}{after['recall']:>10.3f}")

**결과를 읽는 법**: `recall`이 0.893에서 1.000으로 올랐습니다.

`hit`은 "정답 중 하나라도 찾았나"라서 확장 전에도 이미 높았습니다.
`recall`은 "정답을 몇 개나 찾았나"이므로 **여러 조를 봐야 하는 질문에서 확장의 효과가 드러납니다.**
"재택근무일에 하루 몇 시간 일하나요?"처럼 제11조와 제9조를 둘 다 봐야 하는 질문이
확장 덕분에 온전히 채워진 것입니다.

xref 맵을 보면 제11조가 제9조를 참조하고 있죠. **문서가 이미 알려준 연결**을 쓴 것뿐입니다.

> ⚠️ **공짜가 아닙니다.** 확장한 만큼 LLM에게 넘어가는 토큰이 늘고,
> 관련 없는 조가 섞이면 오히려 답이 흐려집니다. `budget`을 두는 이유입니다.
> 실제 프로젝트에서는 "직접검색 > 명시참조 > 부모조항" 순으로 우선순위를 두고
> 토큰 예산 안에서 자릅니다.

## 연습 3. 파서를 깨뜨려보기

**문제**: 공백이 낀 `제 11 조`, 표 안의 `| 제11조`, 두 줄로 잘린 제목을 넣으면 어떻게 되나요?

실제로 넣어봅시다. **먼저 결과를 예측해보고** 실행하세요.

In [ ]:
broken_cases = {
    "정상": "제11조(재택근무)\n① 주 2일 재택근무를 할 수 있다.",
    "조 표지에 공백": "제 11 조 (재택근무)\n① 주 2일 재택근무를 할 수 있다.",
    "표 안에 들어감": "| 제11조(재택근무) | 주 2일 |",
    "제목이 두 줄로 잘림": "제11조(재택\n근무)\n① 주 2일 재택근무를 할 수 있다.",
    "앞에 페이지 번호": "- 3 -\n제11조(재택근무)\n① 주 2일 재택근무를 할 수 있다.",
}

for label, text in broken_cases.items():
    parsed = parse_articles([text])
    if parsed:
        a = parsed[0]
        print(f"  {label:<18} -> {a.number} 제목={a.title!r}")
    else:
        print(f"  {label:<18} -> ❌ 조를 하나도 못 찾음")

**결과 분석**

| 케이스 | 결과 | 이유 |
|---|---|---|
| 조 표지에 공백 | ✅ 잡힘 | 정규식에 `\s*`를 넣어둬서 |
| 표 안에 들어감 | ❌ 실패 | `ARTICLE_RE.match()`는 **줄 맨 앞**부터 봅니다. `|`가 앞에 있으면 안 걸립니다 |
| 제목 두 줄 | ⚠️ 절반 성공 | 조 번호는 잡히지만 **제목이 빈 문자열**이 됩니다 |
| 페이지 번호 | ✅ 잡힘 | 페이지 번호 줄은 그냥 무시되고 다음 줄에서 잡힙니다 |

**제목 두 줄 케이스가 특히 고약합니다.** `제목=''`로 조용히 넘어갑니다.
정규식의 `\(([^)]*)\)` 부분이 한 줄 안에서 닫는 괄호를 못 찾아 매칭에 실패했고,
제목은 선택 항목(`?`)이라 없어도 통과하기 때문입니다.
**에러가 안 나서 더 위험합니다.** 계층 경로가 `제11조`로만 나오고, 검색에서 "재택근무"라는
단어가 프리픽스에서 빠집니다. 무결성 검증이 조 번호만 보기 때문에 이건 잡아내지도 못합니다.

표 케이스를 고쳐봅시다.

In [ ]:
def normalize_line(line: str) -> str:
    """파싱 전에 줄 앞의 표 기호·글머리표를 걷어낸다.

    PDF에서 표를 텍스트로 뽑으면 셀 경계가 |, ┃, 탭 등으로 남는다.
    조 표지가 표 안에 있는 건 규정집에서 드물지 않다 (별표, 개정 대비표 등).
    """
    return re.sub(r"^[\s|｜┃‖\-•·\t]+", "", line)


fixed = "\n".join(normalize_line(line) for line in broken_cases["표 안에 들어감"].split("\n"))
print("정규화 전:", repr(broken_cases["표 안에 들어감"]))
print("정규화 후:", repr(fixed))
parsed = parse_articles([fixed])
print("파싱 결과:", parsed[0].number if parsed else "실패")

### 어디까지 규칙으로 감당하고 어디부터 포기할까

이게 이 연습의 진짜 질문입니다. 정규식은 계속 붙일 수 있지만, 붙일수록
**의도치 않은 것까지 잡기 시작합니다.**

기준을 이렇게 잡으면 좋습니다.

| 상황 | 판단 |
|---|---|
| 대상 문서에서 **자주** 나오고, 규칙이 **단순** | 규칙으로 처리 (공백, 표 기호) |
| 드물게 나오고 규칙이 복잡 | **무결성 검증에 걸리게 두고 수동 확인** |
| 문서마다 형식이 제각각 | LLM 정형 출력을 섞는 것을 검토 |

**포기하는 것도 설계입니다.** 중요한 건 "못 잡았다는 걸 아는 것"이고,
그래서 `check_article_sequence()` 같은 검증이 있는 겁니다.
완벽한 파서보다 **불완전하지만 자기가 실패한 걸 아는 파서**가 낫습니다.

## 연습 4. 고장 난 지표 또 찾아내기

**문제**: 정답이 앞부분에 몰려 있다면? 항상 같은 청크만 돌려주는 검색기의 MRR은?

본문 8-1에서 `k`가 청크 수보다 커서 hit@k가 항상 1.0이 되는 걸 봤습니다.
같은 종류의 함정을 더 찾아봅시다. **지표를 속이는 가짜 검색기**를 만들어서 시험하면 빨리 보입니다.

In [ ]:
def dumb_searcher_always_same(question, k=3):
    """질문을 아예 보지 않고 항상 앞쪽 청크만 돌려주는 가짜 검색기."""
    return [(c, 1.0) for c in chunks[:k]]


def dumb_searcher_random(question, k=3):
    """고정된 순서로 아무거나 돌려주는 가짜 검색기 (재현성을 위해 순서 고정)."""
    import random

    rng = random.Random(42)
    picked = rng.sample(chunks, k)
    return [(c, 1.0) for c in picked]


print(f"{'검색기':<24}{'hit':>8}{'MRR':>8}")
print("-" * 40)
for name, fn in (
    ("진짜 TF-IDF 검색", search),
    ("항상 앞쪽 3개만", dumb_searcher_always_same),
    ("아무거나 3개", dumb_searcher_random),
):
    s = evaluate(fn, questions, k=3)
    print(f"{name:<24}{s['hit']:>8.3f}{s['MRR']:>8.3f}")

**결과를 읽는 법**

바보 검색기들이 0.083을 받았습니다. 12문항 중 1문항을 우연히 맞힌 겁니다.
진짜 검색기는 0.917이니 **약 11배 차이**입니다. 이건 좋은 신호입니다 —
지표가 실제로 검색 능력을 재고 있다는 뜻이니까요.

이걸 **베이스라인(baseline)** 이라고 합니다. 새 방법을 평가할 때 항상 물어야 하는 질문입니다.

> **"아무것도 안 하는 것보다 얼마나 나은가?"**

hit 0.7이 좋은 점수인지 나쁜 점수인지는 그 자체로는 알 수 없습니다.
바보 검색기가 0.65를 받는다면 0.7은 거의 의미가 없고, 0.05를 받는다면 0.7은 훌륭한 점수입니다.
**본문 8-1에서 겪은 hit@5 = 1.000이 딱 전자의 경우**였습니다.
그때 베이스라인을 재봤다면 바보 검색기도 1.000을 받는 걸 보고 바로 알아챘을 겁니다.

이제 정답이 문서 어디에 몰려 있는지도 확인해봅시다.

In [ ]:
# 골든셋의 정답이 문서 어디에 몰려 있는지 확인해봅니다.
positions = {a.number: i for i, a in enumerate(articles)}
gold_positions = [positions[art] for q in questions for art in q["articles"] if art in positions]

print(f"전체 조 개수: {len(articles)}")
print(f"골든셋 정답 조의 위치: {sorted(gold_positions)}")
print(f"평균 위치: {sum(gold_positions) / len(gold_positions):.1f} (한가운데면 {len(articles) / 2:.1f})")

평균 위치가 한가운데와 크게 다르지 않습니다. **위치 편향은 없다고 봐도 됩니다.**
바보 검색기가 0.083밖에 못 받은 것도 이것과 같은 이야기입니다.

만약 평균이 2~3에 몰려 있었다면, "앞쪽 청크만 가져오는" 전략이 점수를 얻습니다.
그러면 그 점수만큼은 검색 능력이 아니라 **골든셋을 잘못 만든 결과**입니다.
이럴 때는 뒤쪽 조를 묻는 질문을 더 넣어 균형을 맞춰야 합니다.

### 지표를 의심하는 체크리스트

본문의 것에 이번 연습에서 얻은 걸 더하면 이렇습니다.

- [ ] `k`가 전체 문서 수보다 작은가?
- [ ] 골든셋 질문이 문서 문구를 그대로 베끼지 않았는가?
- [ ] **아무것도 안 하는 베이스라인은 몇 점인가?**
- [ ] **정답이 특정 위치·특정 문서에 몰려 있지 않은가?**
- [ ] 정답이 없어야 할 질문도 골든셋에 있는가?
- [ ] 재고 싶은 게 recall인가 precision인가? 지금 지표가 그걸 재는가?

## 연습 5. 답변 채점(faithfulness) 설계

**문제**: "찾아온 근거대로 답했는가"를 어떻게 잴까요? 어떤 함정이 있을까요?

검색 평가(`hit@k`)와 **완전히 다른 문제**입니다.
검색은 정답이 딱 정해져 있지만, 답변은 표현이 무한합니다. 그래서 LLM에게 채점을 시킵니다.

### 설계

In [ ]:
FAITHFULNESS_PROMPT = """당신은 AI 답변을 채점하는 심사관입니다.

[근거 자료]
{context}

[질문]
{question}

[AI의 답변]
{answer}

위 답변의 각 문장이 [근거 자료]에서 확인되는지 판정하세요.

판정 기준:
- supported: 근거 자료에 명시적으로 있는 내용
- unsupported: 근거 자료에 없는데 답변이 단정한 내용 (환각)
- irrelevant: 질문과 상관없는 내용

주의: 당신의 배경지식으로 판단하지 마세요. 오직 [근거 자료]에 적혀 있는지만 보세요.
일반적으로 맞는 말이어도 근거 자료에 없으면 unsupported입니다.

각 문장별 판정과 이유를 JSON으로 출력하세요."""

print(FAITHFULNESS_PROMPT[:400], "...")

프롬프트에서 가장 중요한 줄은 이겁니다.

> **"당신의 배경지식으로 판단하지 마세요. 일반적으로 맞는 말이어도 근거 자료에 없으면 unsupported입니다."**

이게 없으면 채점 LLM이 **자기가 아는 노동법 지식으로 채점합니다.**
"연차는 15일"이 회사 규정에 없어도 법으로 맞으니까 supported로 판정해버립니다.
그러면 **환각을 정답으로 채점하는** 평가가 됩니다.

### 함정들

| 함정 | 왜 생기나 | 완화 방법 |
|---|---|---|
| **채점 LLM도 틀린다** | 같은 종류의 모델이 같은 실수를 함 | 답변한 모델과 **다른 모델**로 채점 |
| **자기 편향** | LLM은 자기가 쓴 답변을 후하게 봄 | 위와 같음. 사람 채점과 상관관계를 한 번 확인 |
| **채점 비용** | 질문마다 LLM 호출이 한 번 더 | 전량 말고 표본만. 매일 말고 릴리스 전에 |
| **위치 편향** | 앞에 나온 답변을 더 좋게 봄 | 두 답변 비교 시 순서를 바꿔 두 번 채점 |
| **"모른다"의 함정** | 항상 "모른다"고 답하면 faithfulness 만점 | **답변률과 함께** 봐야 함 |

마지막 항목이 특히 중요합니다. 지표 하나만 보면 시스템이 그 지표에 최적화됩니다.
"근거가 없으면 모른다고 답하라"고 시켰으니, 겁먹은 모델은 다 모른다고 합니다.
그러면 faithfulness는 100%인데 **쓸모는 0%**입니다.

In [ ]:
def evaluate_answers(records):
    """faithfulness는 항상 답변률과 함께 봐야 한다는 걸 보여주는 예시.

    records: [{"answered": bool, "faithful": bool}, ...]
    """
    total = len(records)
    answered = [r for r in records if r["answered"]]
    faithful = [r for r in answered if r["faithful"]]

    answer_rate = len(answered) / total
    faithfulness = len(faithful) / len(answered) if answered else 1.0
    useful_rate = len(faithful) / total  # 실제로 쓸모 있었던 비율

    return answer_rate, faithfulness, useful_rate


scenarios = {
    "겁쟁이 (거의 모른다고 답함)": [{"answered": i < 2, "faithful": True} for i in range(10)],
    "허풍쟁이 (다 답하는데 절반이 환각)": [{"answered": True, "faithful": i < 5} for i in range(10)],
    "균형": [{"answered": i < 8, "faithful": i < 7} for i in range(10)],
}

print(f"{'시나리오':<32}{'답변률':>8}{'충실도':>8}{'실질 유용':>10}")
print("-" * 60)
for name, records in scenarios.items():
    a, f, u = evaluate_answers(records)
    print(f"{name:<32}{a:>8.2f}{f:>8.2f}{u:>10.2f}")

**결과를 읽는 법**: 겁쟁이의 충실도는 1.00으로 만점입니다. 허풍쟁이보다 훨씬 높죠.
하지만 **실질 유용도는 0.20**입니다. 10개 중 2개만 답했으니까요.

지표 하나로는 절대 판단할 수 없습니다. **답변률 × 충실도**를 같이 봐야 합니다.

## 정리

| 연습 | 핵심 |
|---|---|
| 골든셋 늘리기 | 어려운 질문 4종 + **정답 없는 질문**을 반드시 포함 |
| xref 확장 | 문서의 숨은 연결 구조를 쓴다. 단 1홉·예산 제한 |
| 파서 깨뜨리기 | 완벽한 파서보다 **자기 실패를 아는 파서** |
| 고장 난 지표 | **베이스라인을 먼저 재라.** 아무것도 안 하는 것보다 얼마나 나은가 |
| faithfulness | 채점 LLM도 틀린다. 지표 하나만 보면 그 지표에 최적화된다 |

다섯 문제를 관통하는 것: **측정하는 행위 자체를 의심하기.**
지표를 만들면 그 지표가 옳은지 확인해야 하고, 파서를 만들면 실패를 감지할 방법이 있어야 하고,
채점기를 만들면 채점기의 편향을 알아야 합니다.

본문으로 돌아가기: [04_rag_regulation.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/04_rag_regulation/04_rag_regulation.ipynb)